In [0]:
%sql
CREATE DATABASE IF NOT EXISTS bronze;

In [0]:
from pyspark.sql.functions import current_timestamp

table_prefix = "tb"
volume_path = "/Volumes/workspace/default/olist_files/"
files = dbutils.fs.ls(volume_path)

for i in files:
    old_table_name = i.name.replace("olist_", "").replace(".csv", "") \
        .replace("_dataset", "") \
        .replace(".csv", "")
    
    #Exceção para geolocalizacao
    if "geolocation" in old_table_name:
        old_table_name = old_table_name.replace("geolocation", "geolocalizacao")
    
    new_table_name = f"{table_prefix}_{old_table_name}"
    
    df = spark.read.format("csv") \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .load(volume_path + i.name) \
        .withColumn("timestamp_ingestion", current_timestamp())
    
    df.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(f"bronze.{new_table_name}")
        
    print(f"Tabela bronze.{new_table_name} criada com sucesso!")

Tabela bronze.tb_customers criada com sucesso!
Tabela bronze.tb_geolocalizacao criada com sucesso!
Tabela bronze.tb_order_items criada com sucesso!
Tabela bronze.tb_order_payments criada com sucesso!
Tabela bronze.tb_order_reviews criada com sucesso!
Tabela bronze.tb_orders criada com sucesso!
Tabela bronze.tb_products criada com sucesso!
Tabela bronze.tb_sellers criada com sucesso!
Tabela bronze.tb_product_category_name_translation criada com sucesso!


In [0]:
import requests
from pyspark.sql.functions import current_timestamp

dbutils.widgets.text("start_date", "01-01-2016", "Start Date (MM-DD-AAAA)")
dbutils.widgets.text("end_date", "12-31-2018", "End Date (MM-DD-AAAA)")

start_date = dbutils.widgets.get("start_date")
end_date = dbutils.widgets.get("end_date")

url = f"https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)?@dataInicial='{start_date}'&@dataFinalCotacao='{end_date}'&$select=dataHoraCotacao,cotacaoCompra&$format=json"

response = requests.get(url)

if response.status_code != 200:
    raise Exception(f"Error: {response.status_code}")

data = response.json()["value"]

df_dollar = spark.createDataFrame(data) \
    .withColumn("timestamp_ingestion", current_timestamp())

df_dollar.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"bronze.{table_prefix+"_cotacao_dolar"}")

print("Tabela bronze.tb_cotacao_dolar criada com sucesso!")



Tabela bronze.tb_cotacao_dolar criada com sucesso!
